# Titanic：特徴量設計とGradient Boostingによる生存予測

Kaggleの「Titanic: Machine Learning from Disaster」を題材に、乗客情報から生存・死亡を予測する二値分類モデルを構築しました。  
欠損値処理、カテゴリ変数の変換、家族人数の特徴量設計、5分割交差検証、特徴量重要度の分析まで行っています。

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:12px;margin:12px 0;">
<b style="color:#1b5e20;">結果</b><br>
5-Fold CV Accuracy：<b>0.83054</b><br>
Kaggle Public Score（取り組み中の最高値）：<b>0.78468</b>
</div>

### 色の意味

- <span style="color:#2e7d32;"><b>緑</b></span>：最終予測に必要な処理
- <span style="color:#1565c0;"><b>青</b></span>：データやモデルを理解するための分析
- <span style="color:#ef6c00;"><b>黄</b></span>：評価や結果を解釈する際の注意点


## 1. ライブラリとデータの読み込み

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b>データ加工にpandas、モデル構築にscikit-learnを使用します。学習データには正解ラベル<code>Survived</code>があり、テストデータにはありません。
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

DATA_DIR = "/kaggle/input/competitions/titanic"

train_data = pd.read_csv(f"{DATA_DIR}/train.csv")
test_data = pd.read_csv(f"{DATA_DIR}/test.csv")

print("train:", train_data.shape)
print("test :", test_data.shape)
train_data.head()


## 2. データの概要と欠損値

<div style="background:#e3f2fd;border-left:6px solid #1565c0;padding:10px;">
<b>分析用：</b>各列の型と欠損数を確認します。特に<code>Age</code>、<code>Cabin</code>、<code>Embarked</code>に欠損があり、モデルで使う列は補完が必要です。
</div>


In [ ]:
summary = pd.DataFrame({
    "dtype": train_data.dtypes.astype(str),
    "missing": train_data.isnull().sum(),
    "unique": train_data.nunique(),
})

display(summary)
print("Overall survival rate:", train_data["Survived"].mean())


## 3. 性別による生存率の違い

<div style="background:#e3f2fd;border-left:6px solid #1565c0;padding:10px;">
<b>分析用：</b>女性の生存率は約74.2％、男性は約18.9％でした。性別は生存予測に大きく関係すると考えられます。
</div>


In [ ]:
survival_by_sex = (
    train_data.groupby("Sex")["Survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passengers"})
)

survival_by_sex


## 4. 客室クラスによる生存率の違い

<div style="background:#e3f2fd;border-left:6px solid #1565c0;padding:10px;">
<b>分析用：</b>客室クラス<code>Pclass</code>ごとの生存率を確認します。性別だけでなく、乗船位置や社会的背景に関係する客室クラスも重要な特徴量です。
</div>


In [ ]:
survival_by_class = train_data.groupby("Pclass")["Survived"].mean()

survival_by_class.plot(kind="bar", color="#4C78A8")
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

survival_by_class


## 5. 特徴量設計と欠損値処理

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b><code>SibSp</code>（同乗する兄弟・配偶者）と<code>Parch</code>（同乗する親・子）から、本人を含む家族人数<code>FamilySize</code>を作成します。
</div>

<div style="background:#fff3e0;border-left:6px solid #ef6c00;padding:10px;margin-top:10px;">
<b>補完方針：</b><code>Age</code>と<code>Fare</code>は外れ値の影響を受けにくい中央値、<code>Embarked</code>は最頻値で補完します。補完値は学習データから計算し、テストデータにも同じ値を使用します。
</div>


In [ ]:
for data in (train_data, test_data):
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

age_median = train_data["Age"].median()
fare_median = train_data["Fare"].median()
embarked_mode = train_data["Embarked"].mode()[0]

for data in (train_data, test_data):
    data["Age"] = data["Age"].fillna(age_median)
    data["Fare"] = data["Fare"].fillna(fare_median)
    data["Embarked"] = data["Embarked"].fillna(embarked_mode)

features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Embarked",
    "Parch",
    "FamilySize",
]

print(train_data[features].isnull().sum())


## 6. カテゴリ変数のOne-Hot Encoding

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b><code>Sex</code>と<code>Embarked</code>を0・1の列へ変換します。学習データとテストデータの列を揃えることで、カテゴリの種類が異なっても予測できるようにします。
</div>


In [ ]:
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
X_test = X_test.reindex(columns=X.columns, fill_value=0)

y = train_data["Survived"]

print("X:", X.shape)
print("X_test:", X_test.shape)
X.head()


## 7. Gradient Boostingモデル

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b>Gradient Boostingは、前の決定木が間違えたデータを次の木で重点的に学習し、複数の弱いモデルを順番に組み合わせる手法です。
</div>

Random Forestも比較しましたが、このNotebookでは最終的に<code>GradientBoostingClassifier</code>を使用しています。


In [ ]:
model = GradientBoostingClassifier(random_state=1)


## 8. 5分割交差検証

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b>学習データを5つに分け、4つで学習・1つで検証する処理を5回繰り返します。1回の分割だけに依存せず、モデルの安定性を確認します。
</div>


In [7]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
)

print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())


[0.81564246 0.8258427  0.84831461 0.80337079 0.85955056]
0.8305442219571905
平均: 0.8305442219571905
標準偏差: 0.020686792049081817


### 交差検証結果

| Fold | Accuracy |
|---:|---:|
| 1 | 0.8156 |
| 2 | 0.8258 |
| 3 | 0.8483 |
| 4 | 0.8034 |
| 5 | 0.8596 |
| **Mean** | **0.8305** |

標準偏差は約0.0207で、分割によって約2ポイント程度のばらつきがありました。

<div style="background:#fff3e0;border-left:6px solid #ef6c00;padding:10px;margin-top:10px;">
<b>結果の解釈：</b>CV Accuracyは0.8305でしたが、Kaggle Public Scoreの最高値は0.78468でした。検証データとLeaderboardデータの分布差や、特徴量・モデルの汎化性能を考える必要があります。
</div>


## 9. 全学習データで学習して予測

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b>交差検証で性能を確認した後、学習データ全体を使ってモデルを再学習し、テストデータの生存・死亡を予測します。
</div>


In [ ]:
model.fit(X, y)
predictions = model.predict(X_test)

print("Predicted class counts:")
print(pd.Series(predictions).value_counts().sort_index())


## 10. 特徴量重要度

<div style="background:#e3f2fd;border-left:6px solid #1565c0;padding:10px;">
<b>分析用：</b>モデルが予測に使用した特徴量の重要度を確認します。元の結果では、性別、運賃、客室クラス、年齢、家族人数の順に影響が大きくなりました。
</div>


In [ ]:
importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": model.feature_importances_,
    })
    .sort_values("Importance", ascending=True)
)

importance.plot(
    kind="barh",
    x="Feature",
    y="Importance",
    figsize=(8, 5),
    legend=False,
    color="#59A14F",
)
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importance.sort_values("Importance", ascending=False)


## 11. 提出ファイルの作成

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px;">
<b>必要な処理：</b>Kaggle指定の<code>PassengerId</code>と<code>Survived</code>の2列で提出用CSVを保存します。
</div>


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions,
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved:", submission.shape)
submission.head()


## 12. まとめ

Titanicの乗客データを用いて、データ確認、欠損値補完、カテゴリ変数の変換、特徴量設計、モデル学習、交差検証、特徴量重要度の分析、提出ファイル作成まで、一連の分類分析を実施しました。

家族構成を表す<code>FamilySize</code>を作成し、性別・運賃・客室クラス・年齢などと組み合わせました。また、交差検証とKaggleスコアに差が生じた経験から、1つの評価値だけでなく、検証方法と未知データへの汎化性能を確認する重要性を学びました。

### 今後の改善

- 前処理をPipeline化し、各CV fold内で補完値を学習する
- 年齢を客室クラスや敬称ごとに補完する
- 家族単位・チケット単位の特徴量を検証する
- Logistic Regressionなど単純なモデルとも比較する
- StratifiedKFoldでクラス比率を明示的に維持する
